In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_DIR = Path("../data/processed")

dataset = pd.read_parquet(
    DATA_DIR / "aligned_dataset.parquet"
)

flare_catalog = pd.read_parquet(
    DATA_DIR / "solexs_flare_catalog.parquet"
)

print(dataset.shape)
print(flare_catalog.shape)

(43188, 151)
(11, 8)


In [3]:
dataset["DATETIME"] = pd.to_datetime(dataset["DATETIME"])
flare_catalog["peak_time"] = pd.to_datetime(flare_catalog["peak_time"])

In [4]:
LEAD_TIME = pd.Timedelta(minutes=10)

dataset["flare_next_10min"] = 0

In [5]:
for peak in flare_catalog["peak_time"]:

    start = peak - LEAD_TIME

    mask = (
        (dataset["DATETIME"] >= start) &
        (dataset["DATETIME"] < peak)
    )

    dataset.loc[
        mask,
        "flare_next_10min"
    ] = 1

In [6]:
print(dataset["flare_next_10min"].value_counts())

print()

print(
    dataset["flare_next_10min"]
    .value_counts(normalize=True)
)

flare_next_10min
0    40705
1     2483
Name: count, dtype: int64

flare_next_10min
0    0.942507
1    0.057493
Name: proportion, dtype: float64


In [7]:
peak = flare_catalog.iloc[0]["peak_time"]

dataset[
    (dataset["DATETIME"] >= peak - pd.Timedelta(minutes=12)) &
    (dataset["DATETIME"] <= peak + pd.Timedelta(minutes=2))
][
    ["DATETIME", "flare_next_10min"]
].tail(30)

,DATETIME,flare_next_10min


In [8]:
dataset.to_parquet(
    DATA_DIR / "forecast_dataset.parquet",
    index=False,
)

dataset.to_csv(
    DATA_DIR / "forecast_dataset.csv",
    index=False,
)

print("Forecast dataset saved.")

Forecast dataset saved.


In [9]:
print(dataset["flare_next_10min"].value_counts())

flare_next_10min
0    40705
1     2483
Name: count, dtype: int64
